![Built with AI](https://img.shields.io/badge/Built%20with-AI-blue.svg)
 [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/algoritmos-poli/sesiones_presenciales/blob/main/clase8/python-ReadFile.ipynb)

# Arrays y Listas Enlazadas

### Estructuras de Datos y Laboratorio — Universidad de Antioquia

**Curso:** Estructuras de Datos y Persistencia · Ingeniería de Sistemas  
**Unidad:** 2 — Hashing e índices basados en hash  
**Prerrequisito para:** Hash básico, Hashing estático, Hashing dinámico

---

## ¿Para qué sirve este notebook?

Este material es un **repaso autocontenido** de dos estructuras fundamentales que soportan todo el trabajo de indexación que se verá en la Unidad 2. Puede estudiarse de forma independiente antes o después de clase.

Al finalizar este notebook, se espera que pueda:

- Distinguir entre un *array* estático y uno dinámico, y explicar cómo Python implementa sus listas internamente.
- Implementar una lista enlazada simple (*singly linked list*) desde cero.
- Comparar el costo en notación O de las operaciones principales de ambas estructuras.
- Identificar en qué situación conviene usar una u otra estructura.

---

## Contenido

1. [Arrays](#1-arrays)
   - 1.1 Array estático (*Static Array*)
   - 1.2 Array dinámico (*Dynamic Array*)
   - 1.3 El `list` de Python por dentro
2. [Listas Enlazadas (*Linked Lists*)](#2-listas-enlazadas)
   - 2.1 Nodo (*Node*)
   - 2.2 Lista enlazada simple (*Singly Linked List*)
   - 2.3 Lista doblemente enlazada (*Doubly Linked List*)
3. [Comparación y tabla de complejidad](#3-comparacion)
4. [Ejercicios propuestos](#4-ejercicios)


---
## 1. Arrays

### Repaso teórico

Un ***array*** (arreglo) es una colección de elementos almacenados en posiciones **contiguas de memoria**. Esta contigüidad es su característica más importante: permite calcular la dirección de cualquier elemento en tiempo constante usando la fórmula:

```
dirección(i) = dirección_base + i × tamaño_elemento
```

Esto hace que el acceso por índice sea **O(1)** sin importar el tamaño del arreglo.

### 1.1 Array estático (*Static Array*)

Un *static array* tiene un **tamaño fijo** definido en el momento de su creación. Una vez asignado, no puede crecer ni encogerse. Es la forma más básica de arreglo y la que existe en lenguajes como C o Java (con `int[]`).

**Ventajas:**
- Acceso directo por índice: O(1)
- Sin *overhead* de gestión de memoria

**Desventajas:**
- Tamaño fijo: si se llena, no puede crecer
- Inserción o eliminación en el medio requiere desplazar elementos: O(n)


#### Diagrama UML — `StaticArray`

<div align="center">
  <img src="https://raw.githubusercontent.com/DS-UdeA/2026-1/refs/heads/main/clases/clase_06/notebooks/images/static_array.png" alt="StaticArray">
</div>

#### Documentación de la clase `StaticArray`

| Elemento | Descripción |
|---|---|
| **Clase** | `StaticArray` |
| **Propósito** | Simula un arreglo de tamaño fijo con acceso directo por índice |
| `_data` | Lista interna de Python que almacena los elementos |
| `_capacity` | Capacidad máxima del arreglo, definida al crear la instancia |
| `_size` | Número de elementos actualmente almacenados |
| `get(index)` | Retorna el elemento en la posición `index`. Lanza `IndexError` si está fuera de rango |
| `set(index, value)` | Asigna `value` en la posición `index`. Lanza `IndexError` si está fuera de rango |
| `size()` | Retorna el número de elementos almacenados |
| `capacity()` | Retorna la capacidad máxima del arreglo |


In [1]:
# Static Array implementation

class StaticArray:
    """
    A fixed-size array that simulates static memory allocation.
    Once created, its capacity cannot change.
    """

    def __init__(self, capacity: int):
        """
        Initialize the array with a fixed capacity.
        All positions are set to None by default.

        Args:
            capacity (int): Maximum number of elements the array can hold.
        """
        if capacity <= 0:
            raise ValueError("Capacity must be a positive integer.")
        self._capacity = capacity
        self._data = [None] * capacity
        self._size = 0

    def get(self, index: int):
        """Return the element at the given index. O(1)"""
        if index < 0 or index >= self._capacity:
            raise IndexError(f"Index {index} out of range [0, {self._capacity - 1}].")
        return self._data[index]

    def set(self, index: int, value) -> None:
        """Set the element at the given index. O(1)"""
        if index < 0 or index >= self._capacity:
            raise IndexError(f"Index {index} out of range [0, {self._capacity - 1}].")
        if self._data[index] is None:
            self._size += 1
        self._data[index] = value

    def size(self) -> int:
        """Return the number of stored elements."""
        return self._size

    def capacity(self) -> int:
        """Return the maximum capacity of the array."""
        return self._capacity

    def __str__(self) -> str:
        return str(self._data)

#### Ejemplo 1 — Uso básico del `StaticArray`

In [2]:
# Example 1: Basic usage of StaticArray
arr = StaticArray(5)

arr.set(0, 'Alice')
arr.set(1, 'Bob')
arr.set(4, 'Carlos')

print("Array contents:", arr)
print("Element at index 1:", arr.get(1))
print("Size (stored elements):", arr.size())
print("Capacity (max):", arr.capacity())

Array contents: ['Alice', 'Bob', None, None, 'Carlos']
Element at index 1: Bob
Size (stored elements): 3
Capacity (max): 5


#### Ejemplo 2 — ¿Qué pasa cuando se excede la capacidad?

In [3]:
# Example 2: Attempting to access an out-of-range index
try:
    arr.set(10, 'Diana')  # Index 10 does not exist in a capacity-5 array
except IndexError as e:
    print("IndexError caught:", e)

IndexError caught: Index 10 out of range [0, 4].


---
### 1.2 Array dinámico (*Dynamic Array*)

Un *dynamic array* resuelve la limitación de tamaño fijo. Internamente sigue usando un bloque contiguo de memoria, pero cuando se llena, **duplica su capacidad** (*doubling strategy*): reserva un nuevo bloque el doble de grande, copia los elementos y libera el antiguo.

Este proceso se llama ***resizing*** (redimensionamiento). Aunque una operación de *resize* individual cuesta O(n), su costo **amortizado** por inserción es **O(1)**, porque ocurre cada vez menos frecuentemente a medida que el arreglo crece.

> 💡 **Concepto clave — Costo amortizado:** Si se realizan *n* inserciones y el costo total es O(n), entonces el costo *por inserción* es O(1) en promedio. No todas las operaciones son igualmente costosas, pero el promedio es constante.


#### Diagrama UML — `DynamicArray`

<div align="center">
  <img src="https://raw.githubusercontent.com/DS-UdeA/2026-1/refs/heads/main/clases/clase_06/notebooks/images/dynamic_array.png" alt="DynamicArray">
</div>

#### Documentación de la clase `DynamicArray`

| Elemento | Descripción |
|---|---|
| **Clase** | `DynamicArray` |
| **Propósito** | Arreglo que crece automáticamente al llenarse, usando una estrategia de duplicación |
| `_data` | Bloque interno de almacenamiento |
| `_capacity` | Capacidad actual del bloque interno (puede ser mayor que `_size`) |
| `_size` | Número de elementos lógicamente almacenados |
| `append(value)` | Agrega un elemento al final. Amortizado O(1); O(n) cuando ocurre *resize* |
| `get(index)` | Retorna el elemento en `index`. O(1) |
| `set(index, value)` | Asigna `value` en `index`. O(1) |
| `_resize()` | Método privado: duplica la capacidad interna y reubica los elementos. O(n) |


In [4]:
# Dynamic Array implementation

class DynamicArray:
    """
    A resizable array that doubles its capacity when full.
    Models the internal behaviour of Python's built-in list.
    """

    def __init__(self):
        """Initialize with a small internal capacity."""
        self._capacity = 2
        self._size = 0
        self._data = [None] * self._capacity

    def append(self, value) -> None:
        """
        Add an element at the end of the array.
        Triggers a resize (O(n)) if the array is full; amortized O(1).
        """
        if self._size == self._capacity:
            self._resize()
        self._data[self._size] = value
        self._size += 1

    def get(self, index: int):
        """Return the element at the given index. O(1)"""
        if index < 0 or index >= self._size:
            raise IndexError(f"Index {index} out of range [0, {self._size - 1}].")
        return self._data[index]

    def set(self, index: int, value) -> None:
        """Set the element at the given index. O(1)"""
        if index < 0 or index >= self._size:
            raise IndexError(f"Index {index} out of range [0, {self._size - 1}].")
        self._data[index] = value

    def _resize(self) -> None:
        """Double the internal capacity. O(n)"""
        new_capacity = self._capacity * 2
        new_data = [None] * new_capacity
        for i in range(self._size):
            new_data[i] = self._data[i]
        self._data = new_data
        print(f"  [resize] capacity: {self._capacity} → {new_capacity}")
        self._capacity = new_capacity

    def size(self) -> int:
        """Return the number of stored elements."""
        return self._size

    def capacity(self) -> int:
        """Return the current internal capacity."""
        return self._capacity

    def __str__(self) -> str:
        return str(self._data[:self._size])

#### Ejemplo 3 — Observando el *resize* en acción

In [5]:
# Example 3: Watching the resize happen
dyn = DynamicArray()
print(f"Initial capacity: {dyn.capacity()}")
print()

for i in range(1, 10):
    dyn.append(i * 10)
    print(f"  append({i*10:3}) → size={dyn.size()}, capacity={dyn.capacity()}")

print()
print("Final array:", dyn)

Initial capacity: 2

  append( 10) → size=1, capacity=2
  append( 20) → size=2, capacity=2
  [resize] capacity: 2 → 4
  append( 30) → size=3, capacity=4
  append( 40) → size=4, capacity=4
  [resize] capacity: 4 → 8
  append( 50) → size=5, capacity=8
  append( 60) → size=6, capacity=8
  append( 70) → size=7, capacity=8
  append( 80) → size=8, capacity=8
  [resize] capacity: 8 → 16
  append( 90) → size=9, capacity=16

Final array: [10, 20, 30, 40, 50, 60, 70, 80, 90]


---
### 1.3 El `list` de Python por dentro

El tipo `list` de Python **es** un *dynamic array*. Internamente usa un bloque contiguo de memoria y aplica la misma estrategia de duplicación. Por eso:

- `lista[i]` → O(1)  
- `lista.append(x)` → O(1) amortizado  
- `lista.insert(0, x)` → O(n) — debe desplazar todos los elementos

Se puede inspeccionar el comportamiento de memoria real con el módulo `sys`:


In [6]:
# Example 4: Inspecting Python's list internal capacity using sys
import sys

native = []
print(f"{'Elements':>10} | {'Size (bytes)':>14} | {'Allocated slots (approx)':>26}")
print("-" * 58)

for i in range(17):
    native.append(i)
    print(f"{len(native):>10} | {sys.getsizeof(native):>14} | estimated slots ≈ {sys.getsizeof(native) // 8}")

  Elements |   Size (bytes) |   Allocated slots (approx)
----------------------------------------------------------
         1 |             88 | estimated slots ≈ 11
         2 |             88 | estimated slots ≈ 11
         3 |             88 | estimated slots ≈ 11
         4 |             88 | estimated slots ≈ 11
         5 |            120 | estimated slots ≈ 15
         6 |            120 | estimated slots ≈ 15
         7 |            120 | estimated slots ≈ 15
         8 |            120 | estimated slots ≈ 15
         9 |            184 | estimated slots ≈ 23
        10 |            184 | estimated slots ≈ 23
        11 |            184 | estimated slots ≈ 23
        12 |            184 | estimated slots ≈ 23
        13 |            184 | estimated slots ≈ 23
        14 |            184 | estimated slots ≈ 23
        15 |            184 | estimated slots ≈ 23
        16 |            184 | estimated slots ≈ 23
        17 |            248 | estimated slots ≈ 31


---
## 2. Listas Enlazadas (*Linked Lists*)

### Repaso teórico

Una ***linked list*** (lista enlazada) almacena elementos en **nodos** (*nodes*) dispersos en memoria, donde cada nodo guarda el dato y un puntero (*pointer* o referencia) al siguiente nodo.

A diferencia del *array*, los nodos **no están contiguos en memoria**. Esto tiene consecuencias directas:

| Característica | Array | Linked List |
|---|---|---|
| Acceso por índice | O(1) — directo | O(n) — hay que recorrer |
| Inserción al inicio | O(n) — desplaza todo | O(1) — solo redirige punteros |
| Inserción al final | O(1) amortizado | O(n) sin tail / O(1) con tail |
| Memoria | Contigua y compacta | Dispersa, overhead por punteros |

### ¿Por qué importa esto para hashing?

Las tablas *hash* con *encadenamiento separado* (*separate chaining*) usan listas enlazadas dentro de cada *bucket* para manejar colisiones. Entender el costo de insertar y buscar en una lista enlazada es **prerequisito directo** para analizar el costo de esas tablas.


---
### 2.1 Nodo (*Node*)

El bloque constructor de toda lista enlazada. Cada nodo contiene un dato y una referencia al siguiente nodo.


#### Diagrama UML — `Node` y `SinglyLinkedList`

<div align="center">
  <img src="https://raw.githubusercontent.com/DS-UdeA/2026-1/refs/heads/main/clases/clase_06/notebooks/images/singly_LL.png" alt="Node y SinglyLinkedList">
</div>


In [7]:
# Node — the building block of all linked structures

class Node:
    """
    A single node in a linked list.

    Attributes:
        data: The value stored in this node.
        next (Node): Reference to the next node, or None if this is the last node.
    """

    def __init__(self, data):
        self.data = data
        self.next = None

    def __repr__(self) -> str:
        return f"Node({self.data})"

---
### 2.2 Lista enlazada simple (*Singly Linked List*)

En una lista enlazada simple, cada nodo apunta **solo al siguiente**. El recorrido solo puede hacerse en una dirección: de cabeza (*head*) hacia el final.

#### Documentación de la clase `SinglyLinkedList`

| Elemento | Descripción |
|---|---|
| **Clase** | `SinglyLinkedList` |
| **Propósito** | Lista enlazada en una sola dirección, útil como base de *buckets* en tablas hash |
| `_head` | Referencia al primer nodo de la lista. `None` si la lista está vacía |
| `_size` | Número de elementos almacenados |
| `prepend(data)` | Inserta al inicio. **O(1)** |
| `append(data)` | Inserta al final. **O(n)** — recorre hasta el último nodo |
| `search(data)` | Busca un valor. **O(n)** en el peor caso |
| `delete(data)` | Elimina el primer nodo con ese valor. **O(n)** |
| `size()` | Retorna el número de elementos. **O(1)** |


In [8]:
# Singly Linked List implementation

class SinglyLinkedList:
    """
    A singly linked list where each node points only to the next one.
    Useful as the bucket structure in separate-chaining hash tables.
    """

    def __init__(self):
        """Initialize an empty list."""
        self._head = None
        self._size = 0

    def prepend(self, data) -> None:
        """
        Insert a new node at the beginning of the list. O(1)

        Args:
            data: The value to insert.
        """
        new_node = Node(data)
        new_node.next = self._head
        self._head = new_node
        self._size += 1

    def append(self, data) -> None:
        """
        Insert a new node at the end of the list. O(n)

        Args:
            data: The value to insert.
        """
        new_node = Node(data)
        if self._head is None:
            self._head = new_node
        else:
            current = self._head
            while current.next is not None:
                current = current.next
            current.next = new_node
        self._size += 1

    def search(self, data) -> bool:
        """
        Check whether a value exists in the list. O(n)

        Args:
            data: The value to search for.

        Returns:
            True if found, False otherwise.
        """
        current = self._head
        while current is not None:
            if current.data == data:
                return True
            current = current.next
        return False

    def delete(self, data) -> bool:
        """
        Remove the first node with the given value. O(n)

        Args:
            data: The value to remove.

        Returns:
            True if the node was found and removed, False otherwise.
        """
        if self._head is None:
            return False
        if self._head.data == data:
            self._head = self._head.next
            self._size -= 1
            return True
        current = self._head
        while current.next is not None:
            if current.next.data == data:
                current.next = current.next.next
                self._size -= 1
                return True
            current = current.next
        return False

    def size(self) -> int:
        """Return the number of elements in the list."""
        return self._size

    def __str__(self) -> str:
        """Return a visual representation of the list."""
        nodes = []
        current = self._head
        while current is not None:
            nodes.append(str(current.data))
            current = current.next
        return " → ".join(nodes) + " → None"

#### Ejemplo 5 — Operaciones básicas sobre `SinglyLinkedList`

In [9]:
# Example 5: Basic operations on a SinglyLinkedList
sll = SinglyLinkedList()

sll.append(10)
sll.append(20)
sll.append(30)
sll.prepend(5)   # Inserted at the front in O(1)

print("List after insertions:", sll)
print("Size:", sll.size())
print()

print("Search 20:", sll.search(20))
print("Search 99:", sll.search(99))
print()

sll.delete(20)
print("List after deleting 20:", sll)

List after insertions: 5 → 10 → 20 → 30 → None
Size: 4

Search 20: True
Search 99: False

List after deleting 20: 5 → 10 → 30 → None


#### Ejemplo 6 — Simulando un *bucket* de tabla hash con `SinglyLinkedList`

Este ejemplo anticipa el uso concreto que se le dará a esta estructura en la siguiente clase:

In [10]:
# Example 6: Using a SinglyLinkedList as a hash table bucket
# This previews the separate chaining technique covered in Block 2.

N = 4  # Number of buckets
table = [SinglyLinkedList() for _ in range(N)]

def hash_function(key: int, n: int) -> int:
    """Simple modulo hash function."""
    return key % n

keys = [5, 3, 8, 10, 9, 7]
for key in keys:
    bucket_index = hash_function(key, N)
    table[bucket_index].prepend(key)
    print(f"  key={key} → bucket[{bucket_index}]")

print()
print("Hash table state:")
for i, bucket in enumerate(table):
    print(f"  bucket[{i}]: {bucket}")

  key=5 → bucket[1]
  key=3 → bucket[3]
  key=8 → bucket[0]
  key=10 → bucket[2]
  key=9 → bucket[1]
  key=7 → bucket[3]

Hash table state:
  bucket[0]: 8 → None
  bucket[1]: 9 → 5 → None
  bucket[2]: 10 → None
  bucket[3]: 7 → 3 → None


---
### 2.3 Lista doblemente enlazada (*Doubly Linked List*)

En una *doubly linked list*, cada nodo tiene **dos punteros**: uno hacia el siguiente y otro hacia el anterior. Esto permite recorrer la lista en ambas direcciones y hace que la eliminación de un nodo conocido sea O(1) en lugar de O(n).


#### Diagrama UML — `DoublyNode` y `DoublyLinkedList`

<div align="center">
  <img src="https://raw.githubusercontent.com/DS-UdeA/2026-1/refs/heads/main/clases/clase_06/notebooks/images/doubly_LL.png" alt="DpublyNode y DoublyLinkedList">
</div>


In [11]:
# Doubly Linked List implementation

class DoublyNode:
    """
    A node with references to both the next and previous nodes.

    Attributes:
        data: Value stored in the node.
        next (DoublyNode): Reference to the next node.
        prev (DoublyNode): Reference to the previous node.
    """
    def __init__(self, data):
        self.data = data
        self.next = None
        self.prev = None


class DoublyLinkedList:
    """
    A doubly linked list supporting O(1) insertion at both ends
    and O(1) deletion when the target node is already known.
    """

    def __init__(self):
        self._head = None
        self._tail = None
        self._size = 0

    def append(self, data) -> None:
        """Insert at the end. O(1) thanks to the tail pointer."""
        new_node = DoublyNode(data)
        if self._tail is None:
            self._head = self._tail = new_node
        else:
            new_node.prev = self._tail
            self._tail.next = new_node
            self._tail = new_node
        self._size += 1

    def prepend(self, data) -> None:
        """Insert at the beginning. O(1)"""
        new_node = DoublyNode(data)
        if self._head is None:
            self._head = self._tail = new_node
        else:
            new_node.next = self._head
            self._head.prev = new_node
            self._head = new_node
        self._size += 1

    def delete_node(self, node: DoublyNode) -> None:
        """
        Delete a node given a direct reference to it. O(1)
        This is only possible because we have the prev pointer.

        Args:
            node (DoublyNode): The node to remove.
        """
        if node.prev:
            node.prev.next = node.next
        else:
            self._head = node.next
        if node.next:
            node.next.prev = node.prev
        else:
            self._tail = node.prev
        self._size -= 1

    def search(self, data):
        """Return the first node containing data, or None. O(n)"""
        current = self._head
        while current:
            if current.data == data:
                return current
            current = current.next
        return None

    def size(self) -> int:
        return self._size

    def __str__(self) -> str:
        nodes = []
        current = self._head
        while current:
            nodes.append(str(current.data))
            current = current.next
        return " ↔ ".join(nodes) if nodes else "Empty"

#### Ejemplo 7 — Eliminación O(1) con referencia directa al nodo

In [12]:
# Example 7: O(1) deletion using a direct node reference
dll = DoublyLinkedList()
for val in [10, 20, 30, 40, 50]:
    dll.append(val)

print("Initial list:", dll)

# Find the node containing 30
target = dll.search(30)
print(f"Found node: {target.data}")

# Delete it in O(1) — no traversal needed
dll.delete_node(target)
print("After deleting 30:", dll)

Initial list: 10 ↔ 20 ↔ 30 ↔ 40 ↔ 50
Found node: 30
After deleting 30: 10 ↔ 20 ↔ 40 ↔ 50


---
## 3. Comparación y tabla de complejidad

### Resumen de complejidad — Notación O

La siguiente tabla resume el costo de las operaciones fundamentales para cada estructura vista en este bloque.

| Operación | Static Array | Dynamic Array | Singly Linked List | Doubly Linked List |
|---|:---:|:---:|:---:|:---:|
| Acceso por índice | **O(1)** | **O(1)** | O(n) | O(n) |
| Búsqueda | O(n) | O(n) | O(n) | O(n) |
| Inserción al inicio | O(n) | O(n) | **O(1)** | **O(1)** |
| Inserción al final | O(1)* | O(1) amort. | O(n) / O(1)** | **O(1)** |
| Inserción en medio | O(n) | O(n) | O(n) | O(n) |
| Eliminación al inicio | O(n) | O(n) | **O(1)** | **O(1)** |
| Eliminación (nodo conocido) | O(n) | O(n) | O(n) | **O(1)** |
| Memoria extra por elemento | Ninguna | Ninguna | 1 puntero | 2 punteros |

<br>

> *\* Solo si hay espacio disponible en el array fijo.*  
> *\*\* O(1) si se mantiene un puntero `tail`.*

### ¿Qué estructura usar y cuándo?

| Situación | Estructura recomendada |
|---|---|
| Acceso frecuente por índice | Array (estático o dinámico) |
| Inserciones/eliminaciones frecuentes al inicio | Linked list |
| Tamaño desconocido en tiempo de compilación | Dynamic array |
| *Buckets* en una tabla hash | Singly linked list (por su inserción O(1) al inicio) |
| Necesidad de recorrido bidireccional | Doubly linked list |


---
## 4. Ejercicios propuestos

Los siguientes ejercicios están diseñados para verificar la comprensión del material. Se recomienda resolverlos antes de la siguiente clase.


In [ ]:
# Exercise 1
# Implement a method reverse() for SinglyLinkedList that reverses
# the list in-place without using any auxiliary data structure.
# What is the time complexity of your solution?

# Write your solution here


In [ ]:
# Exercise 2
# Given the following hash table using SinglyLinkedList buckets:
#
#   N = 5  (5 buckets)
#   Keys to insert: [14, 3, 22, 9, 19, 11, 7]
#
# a) Insert all keys using h(k) = k % N
# b) Print the state of each bucket after all insertions
# c) Count how many collisions occurred (more than one key per bucket)

# Write your solution here


In [ ]:
# Exercise 3 (challenge)
# Modify DynamicArray to add a method remove(index) that:
# - Removes the element at the given index
# - Shifts the remaining elements left to fill the gap
# - Returns the removed value
# What is the time complexity? Justify your answer.

# Write your solution here


---
## Referencias de este bloque

- Goodrich, M. T., Tamassia, R., & Goldwasser, M. H. (2013). *Data Structures and Algorithms in Python*. Wiley. **Capítulos 3, 5 y 7.**
- Bhargava, A. (2016). *Grokking Algorithms*. Manning. **Capítulos 1 y 2.**
- Visualizador interactivo de listas enlazadas: [VisuAlgo — Linked List](https://visualgo.net/en/list)
- Documentación oficial de Python 3 — [Built-in Types: list](https://docs.python.org/3/library/stdtypes.html#list)


---

> 🤖 **AI Disclosure:** 
> This document was created with the assistance of Artificial Intelligence language models. The content has been reviewed, edited, and validated by a human author to ensure accuracy and quality.